# Exceptional Model Mining

Exceptional Model Mining (EMM) generalizes subgroup discovery to several target
attributes: a **model** is fitted on the target attributes in the subgroup and
in the rest of the data, and the quality measures how much the two models
differ. pysubgroup's `emm` package implements the model classes of
Duivesteijn, Feelders, and Knobbe: *Exceptional Model Mining* (DMKD 30, 2016):

| Model class | Target | Quality functions |
| --- | --- | --- |
| correlation | `CorrelationTarget(x, y)` | `CorrelationDifferenceQF` (`phi_abs`, `phi_ent`), `CorrelationSignificanceQF` (`phi_scd`) |
| simple regression | `RegressionTarget(x, y)` | `SlopeSignificanceQF` (`phi_ssd`) |
| regression | `GeneralRegressionTarget(*predictors, response)` | `CooksDistanceQF` (`phi_Cook`) |
| logistic regression | `LogisticRegressionTarget(x, y, value)` | `LogisticInteractionQF` (`phi_sed`) |
| association | `AssociationTarget(x, y)` | `AssociationDevianceQF` (`phi_dev`) |
| decision table | `DecisionTableTarget(*inputs, output)` | `HellingerQF`, `BDeuQF` |
| any model with a likelihood | `LikelihoodTarget(model)`, e.g. with `PolynomialRegressionModel` | `LikelihoodQF` |

The EMM targets are searched with the standard algorithms. Every example below takes
the same steps on the **Adult** census data: fit the target on the model
attributes, define the task (search space, quality function, result size,
depth, and a minimum size of 200 rows), run a search, and show the results with
`to_dataframe`.

In [ ]:
from sklearn.datasets import fetch_openml

# Adult / Census Income (OpenML dataset 1590): 48,842 people, income above 50K or not.
adult = fetch_openml("adult", version=2, as_frame=True)
# fnlwgt is a census sampling weight, not a property of the person.
data = adult.frame.drop(columns=["fnlwgt"])
features = data.drop(columns="class")
data.head()

In [ ]:
import pandas as pd

import pysubgroup as sd
from pysubgroup import visualization as viz

# The model attributes stay out of the descriptions.
model_attributes = ["age", "hours-per-week", "education-num", "sex"]
conditions = sd.generate_conditions(features.drop(columns=model_attributes), n_bins=5)
print(len(conditions), "conditions")

## Correlation

Overall, age and weekly working hours are hardly correlated. Where does the
correlation differ most from the rest of the data? `CorrelationDifferenceQF`
(`phi_abs`) is the absolute difference of the correlations. It favors small
subgroups; `entropy=True` (`phi_ent`) weights it by the entropy of the split
into subgroup and complement.

In [ ]:
correlation = sd.CorrelationTarget().fit(data, "age", "hours-per-week")
print(sd.to_dataframe([sd.Rule()], data, correlation)[["size", "correlation"]])

task = sd.SubgroupDiscoveryTask(
    data, correlation, conditions, sd.CorrelationDifferenceQF(entropy=True),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, correlation).round(4)

The best subgroup against its complement: each cloud of points gets its own
correlation in the legend.

In [ ]:
viz.plot_model_scatter(found, data, correlation, subgroup=1, lines=False);

`CorrelationSignificanceQF` (`phi_scd`) tests whether the two correlations
differ (Fisher z). With `score="statistic"` it ranks by the test statistic `|z|`
instead of `1 - p`, which rounds to 1 for strong effects and makes many
subgroups tie.

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, correlation, conditions, sd.CorrelationSignificanceQF(score="statistic"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, correlation).round(4)

The statistics behind the test for the best subgroup: the correlation and the
mean working hours of the younger and older half, in the subgroup and in the
complement.

In [ ]:
members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement

pd.DataFrame({
    name: {
        "rows": len(part),
        "correlation": part["age"].corr(part["hours-per-week"]),
        "mean hours, age <= 37": part.loc[part["age"] <= 37, "hours-per-week"].mean(),
        "mean hours, age > 37": part.loc[part["age"] > 37, "hours-per-week"].mean(),
    }
    for name, part in {"subgroup": members, "complement": rest}.items()
}).round(3)

## Simple regression

`SlopeSignificanceQF` (`phi_ssd`) tests whether the regression slope of working
hours on age differs between subgroup and complement. The scatter plot shows the
best subgroup against its complement with both regression lines (DMKD Figs. 3
and 4):

In [ ]:
regression = sd.RegressionTarget().fit(data, "age", "hours-per-week")
task = sd.SubgroupDiscoveryTask(
    data, regression, conditions, sd.SlopeSignificanceQF(score="statistic"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(
    found, data, regression,
    ["size", "slope", "intercept", "slope_complement", "intercept_complement"],
).round(4)

In [ ]:
viz.plot_model_scatter(found, data, regression, subgroup=1);

## Regression with several predictors

`CooksDistanceQF` (`phi_Cook`) measures how far the regression coefficients of
the subgroup are from those on the whole dataset, here for working hours on age
and years of education.

In [ ]:
general = sd.GeneralRegressionTarget().fit(data, "age", "education-num", "hours-per-week")
task = sd.SubgroupDiscoveryTask(
    data, general, conditions, sd.CooksDistanceQF(),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, general, ["size", "coefficients"]).round(4)

The coefficients of the best subgroup next to those on the whole dataset
(`Rule()` is the empty description):

In [ ]:
sd.to_dataframe([sd.Rule(), found[0]], data, general, ["size", "coefficients"]).drop(
    columns="quality"
).round(4)

## Logistic regression

`LogisticInteractionQF` (`phi_sed`) fits `logit P(>50K) = b0 + b1 D + b2 age +
b3 D age` with the subgroup indicator `D` and tests the interaction `b3`: in
which subgroups does income depend differently on age? Each subgroup needs its
own fit, so we use beam search here.

In [ ]:
logistic = sd.LogisticRegressionTarget().fit(data, "age", "class", ">50K")
task = sd.SubgroupDiscoveryTask(
    data, logistic, conditions, sd.LogisticInteractionQF(score="statistic"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.BeamSearch(beam_width=10).execute(task)
sd.to_dataframe(found, data, logistic, ["size", "logit_slopes"]).round(4)

The share of people earning more than 50K by age group, in the best subgroup and
its complement. Up to middle age, the complement's share rises from about 1% to
26% and the subgroup's from about 10% to 50%. On the logit scale the
complement's increase is the steeper one, which the logit slopes in the table
summarize as straight lines (they ignore the decline after 55).

In [ ]:
members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement

age_group = pd.cut(data["age"], bins=range(15, 95, 10))
shares = pd.DataFrame({
    "subgroup": members["class"].eq(">50K").groupby(age_group[members.index], observed=True).mean(),
    "complement": rest["class"].eq(">50K").groupby(age_group[rest.index], observed=True).mean(),
})
shares.plot(marker="o", xlabel="age", ylabel="share of >50K", rot=45);

## Any model with a likelihood

`LikelihoodTarget` takes a pluggable model class (`LikelihoodModel`) and
`LikelihoodQF` scores a subgroup with the likelihood-ratio test of separate
models on subgroup and complement against one model on the whole dataset:
`LR = 2 [l_G + l_C - l_D]`, asymptotically chi-squared with as many degrees of
freedom as the model has parameters. With `score="statistic"` it ranks by `LR`.

The built-in model class is `PolynomialRegressionModel`, here a quadratic
regression of working hours on age (with the error variance estimated, four
parameters). The fit on the dataset, then a beam search (each subgroup needs
its own fit):

In [ ]:
quadratic = sd.LikelihoodTarget(sd.PolynomialRegressionModel(degree=2)).fit(
    data, "age", "hours-per-week"
)
print(sd.to_dataframe([sd.Rule()], data, quadratic).round(4))

task = sd.SubgroupDiscoveryTask(
    data, quadratic, conditions, sd.LikelihoodQF(score="statistic"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.BeamSearch(beam_width=10).execute(task)
sd.to_dataframe(found, data, quadratic, ["size", "parameters", "parameters_complement"]).round(4)

Mean working hours per age in the best subgroup and its complement, with the
fitted parabolas. Husbands work longer at every age; their curve rises less
steeply in the twenties and stays near its peak of about 46 hours until 60:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement
fits = sd.to_dataframe(found[:1], data, quadratic, ["parameters", "parameters_complement"])
ages = np.linspace(data["age"].min(), data["age"].max(), 200)

fig, ax = plt.subplots()
for name, part, suffix in (("subgroup", members, ""), ("complement", rest, "_complement")):
    means = part.groupby("age")["hours-per-week"].mean()
    points = ax.scatter(means.index, means, s=10, label=f"{name}: mean per age")
    b0, b1, b2 = (fits[f"b{j}{suffix}"].iloc[0] for j in range(3))
    color = points.get_facecolors()[0]
    ax.plot(ages, b0 + b1 * ages + b2 * ages**2, color=color, label=f"{name}: fit")
ax.set(xlabel="age", ylabel="hours per week", title=sd.utils.rule_label(found[0][1]))
ax.legend();

### Your own model class

A model class implements `prepare` (store the model attributes), `n_parameters`,
`fit(mask)` (the maximum-likelihood parameters on the rows `mask`, or `None`),
`log_likelihood(parameters, mask)` (one value per row), and optionally
`describe` (named parameters for `to_dataframe`). This one models working hours
with a normal distribution, so the search finds subgroups whose mean or spread
of working hours differs from the rest:

In [ ]:
from scipy import stats


class NormalModel(sd.LikelihoodModel):
    """One numeric attribute with a normal distribution (mean and variance)."""

    def prepare(self, data, attributes):
        (name,) = attributes
        self.values_ = data[name].to_numpy(dtype=float)

    @property
    def n_parameters(self):
        return 2

    def fit(self, mask):
        values = self.values_[mask]
        if len(values) < 2 or values.var() == 0:
            return None
        return values.mean(), values.var()

    def log_likelihood(self, parameters, mask=None):
        values = self.values_ if mask is None else self.values_[mask]
        mean, variance = parameters
        return stats.norm.logpdf(values, mean, np.sqrt(variance))

    def describe(self, parameters):
        mean, variance = (np.nan, np.nan) if parameters is None else parameters
        return {"mean": mean, "std": np.sqrt(variance)}


hours = sd.LikelihoodTarget(NormalModel()).fit(data, "hours-per-week")
task = sd.SubgroupDiscoveryTask(
    data, hours, conditions, sd.LikelihoodQF(score="statistic"),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, hours, ["size", "parameters", "parameters_complement"]).round(3)

`LikelihoodQF("density")` is pysubgroup 0.x's `EMM_Likelihood` instead: the mean
likelihood of the subgroup's model on the subgroup minus on the complement
(with `PolynomialRegressionModel(1, variance=1.0)` it gives pysubgroup 0.x's
results). It has no publication and depends on the scale of the attributes,
so the likelihood-ratio test is the default.

## Nominal models: association and decision tables

`AssociationDevianceQF` (`phi_dev`) asks where the association between two
nominal attributes, here sex and income, differs from the rest of the data.
`HellingerQF` compares the distribution of an output given its inputs (a
decision table, here income given sex) between subgroup and complement.
It has no size term, so Leman et al. raise the minimum support instead; here
it is 2% of the rows.

In [ ]:
association = sd.AssociationTarget().fit(data, "sex", "class")
task = sd.SubgroupDiscoveryTask(
    data, association, conditions, sd.AssociationDevianceQF(),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(200)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, association).round(4)

The income distribution per sex in the best subgroup and in its complement:

In [ ]:
members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement

pd.concat({
    "subgroup": pd.crosstab(members["sex"], members["class"], normalize="index"),
    "complement": pd.crosstab(rest["sex"], rest["class"], normalize="index"),
}).round(3)

In [ ]:
decision_table = sd.DecisionTableTarget().fit(data, "sex", "class")
task = sd.SubgroupDiscoveryTask(
    data, decision_table, conditions, sd.HellingerQF(),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(len(data) // 50)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, decision_table).round(4)

`subgroup_data` returns the rows of the best subgroup; the complement is the rest
of the data. Their output distributions per input value (income per sex):

In [ ]:
members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement

pd.concat({
    "subgroup": pd.crosstab(members["sex"], members["class"], normalize="index"),
    "complement": pd.crosstab(rest["sex"], rest["class"], normalize="index"),
}).round(3)

The best subgroup, a capital gain above 7,688, only says that almost everyone
in it earns more than 50K. `HellingerQF(entropy=True)` weights the distance
with the entropy of the split between subgroup and complement (`phi_ef`, as
DMKD Sect. 3.2.1 suggests for quality measures; this weighted `phi_Hel` is a
pysubgroup extension), which favours larger subgroups:

In [ ]:
task = sd.SubgroupDiscoveryTask(
    data, decision_table, conditions, sd.HellingerQF(entropy=True),
    result_set_size=5, depth=2, filters=[sd.MinSizeFilter(len(data) // 50)],
)
found = sd.DFS().execute(task)
sd.to_dataframe(found, data, decision_table).round(4)

Among married people, women and men earn more than 50K equally often; in the
complement, men do so almost twice as often as women:

In [ ]:
members = sd.subgroup_data(found[0], data)
rest = data.drop(members.index)  # the complement

pd.concat({
    "subgroup": pd.crosstab(members["sex"], members["class"], normalize="index"),
    "complement": pd.crosstab(rest["sex"], rest["class"], normalize="index"),
}).round(3)

See `user_guide/09-exceptional-model-mining.md` for the options (`reference`, `score`),
the deviations from the papers, and the checks against the papers' examples.